In [1]:
import pandas as pd
import numpy as np

# Load Zillow data


In [3]:
df= pd.read_csv('zillow1.csv')
df.head()

,RegionID,SizeRank,RegionName,RegionType,StateName,State,City,Metro,CountyName,2000-01-31,...,2025-04-30,2025-05-31,2025-06-30,2025-07-31,2025-08-31,2025-09-30,2025-10-31,2025-11-30,2025-12-31,2026-01-31
0,91982,1,77494,zip,TX,TX,Katy,"Houston-The Woodlands-Sugar Land, TX",Fort Bend County,204988.368604,...,486136.109041,484000.352719,481473.267183,479455.235388,478365.621634,478562.451979,479293.472947,480341.632597,480930.867232,480543.633571
1,61148,2,8701,zip,NJ,NJ,Lakewood,"New York-Newark-Jersey City, NY-NJ-PA",Ocean County,111719.941374,...,529389.748384,532145.350705,534655.041351,536426.444429,537661.425723,540194.582941,545049.835219,550925.682161,557122.838698,561559.028597
2,91940,3,77449,zip,TX,TX,Katy,"Houston-The Woodlands-Sugar Land, TX",Harris County,102728.188083,...,274073.754470,273277.617577,272287.601139,271356.056827,270430.150384,269659.092826,268926.643529,268297.386452,267961.720785,267417.889737
3,62080,4,11368,zip,NY,NY,New York,"New York-Newark-Jersey City, NY-NJ-PA",Queens County,172864.490023,...,528122.229315,528512.439633,530570.039974,533309.674869,534970.696579,536204.071877,537227.467932,539139.357023,541823.178015,546584.297355
4,91733,5,77084,zip,TX,TX,Houston,"Houston-The Woodlands-Sugar Land, TX",Harris County,102246.148663,...,270027.619313,269137.452625,268139.369611,267230.225642,266357.802018,265483.654722,264636.773995,263910.190406,263568.203302,263066.097594


In [6]:
# Reorganize data:
import pandas as pd

date_cols = [c for c in df_zip.columns if "-" in c]  # date columns like 2013-01-31

panel_zip = df_zip.melt(
    id_vars=["RegionID", "RegionName", "State", "City", "CountyName"],
    value_vars=date_cols,
    var_name="date",
    value_name="price"
)

panel_zip["date"] = pd.to_datetime(panel_zip["date"])
panel_zip["year"] = panel_zip["date"].dt.year
panel_zip["month"] = panel_zip["date"].dt.month

In [7]:

date_cols = [c for c in df_zip.columns if "-" in c]  # date columns like 2013-01-31

panel_zip = df_zip.melt(
    id_vars=["RegionID", "RegionName", "State", "City", "CountyName"],
    value_vars=date_cols,
    var_name="date",
    value_name="price"
)

panel_zip["date"] = pd.to_datetime(panel_zip["date"])
panel_zip["year"] = panel_zip["date"].dt.year
panel_zip["month"] = panel_zip["date"].dt.month

In [8]:
panel_zip = panel_zip.rename(columns={"RegionName": "zip_code"})
panel_zip["zip_code"] = panel_zip["zip_code"].astype(str).str.zfill(5)

In [9]:

panel_zip = panel_zip[(panel_zip["year"] >= 2013) & (panel_zip["year"] <= 2023)].copy()

In [11]:
zip_year = (
    panel_zip
    .groupby(
        ["RegionID", "zip_code", "State", "City", "CountyName", "year"],
        as_index=False
    )
    .agg({"price": "mean"})
)

zip_year.head()

,RegionID,zip_code,State,City,CountyName,year,price
0,58196,01001,MA,Agawam,Hampden County,2013,179159.306814
1,58196,01001,MA,Agawam,Hampden County,2014,179950.420804
2,58196,01001,MA,Agawam,Hampden County,2015,181797.078539
3,58196,01001,MA,Agawam,Hampden County,2016,190231.359754
4,58196,01001,MA,Agawam,Hampden County,2017,197659.904419


In [12]:
zip_year_ca = zip_year[zip_year["State"] == "CA"]
zip_year_ca.head()

,RegionID,zip_code,State,City,CountyName,year,price
250008,95982,90001,CA,Florence-Graham,Los Angeles County,2013,184912.712009
250009,95982,90001,CA,Florence-Graham,Los Angeles County,2014,225886.255973
250010,95982,90001,CA,Florence-Graham,Los Angeles County,2015,253188.792513
250011,95982,90001,CA,Florence-Graham,Los Angeles County,2016,273868.654666
250012,95982,90001,CA,Florence-Graham,Los Angeles County,2017,315517.525062


# Load WFH data

In [13]:
import pandas as pd
import glob
import re
from pathlib import Path

folder = r"C:\Users\minht\Capstone new\WFHnew"

files = glob.glob(str(Path(folder) / "ACSST5Y*.S0801-Data.csv"))

dfs = []

for f in files:
    
    year = int(re.search(r"ACSST5Y(\d{4})\.S0801-Data\.csv", Path(f).name).group(1))

    d = pd.read_csv(
        f,
        usecols=["GEO_ID", "NAME", "S0801_C01_003E"],
        dtype=str,
        low_memory=False
    )

    d = d.rename(columns={"S0801_C01_003E": "wfh_share"})
    d["year"] = year

    dfs.append(d)

acs_wfh = pd.concat(dfs, ignore_index=True)

# Convert to numeric
acs_wfh["wfh_share"] = pd.to_numeric(acs_wfh["wfh_share"], errors="coerce")

# Extract ZIP from NAME like "ZCTA5 90001"
acs_wfh["zip"] = acs_wfh["NAME"].str.extract(r"(\d{5})")

# optional: keep only rows where ZIP exists
acs_wfh = acs_wfh[acs_wfh["zip"].notna()].copy()

acs_wfh["zip"] = acs_wfh["zip"].astype(str).str.zfill(5)
acs_wfh.head()


,GEO_ID,NAME,wfh_share,year,zip
1,8600000US89010,ZCTA5 89010,61.8,2013,89010
2,8600000US89019,ZCTA5 89019,57.8,2013,89019
3,8600000US89060,ZCTA5 89060,76.7,2013,89060
4,8600000US89061,ZCTA5 89061,75.7,2013,89061
5,8600000US89439,ZCTA5 89439,75.6,2013,89439


## Load population data

In [14]:
import pandas as pd
import glob
import re
from pathlib import Path

folder = r"C:\Users\minht\Capstone new\Popnew"

files = glob.glob(str(Path(folder) / "ACSDT5Y*.B01003-Data.csv"))

dfs = []

for f in files:
    
    year = int(re.search(r"ACSDT5Y(\d{4})\.B01003-Data\.csv", Path(f).name).group(1))

    d = pd.read_csv(
        f,
        usecols=["GEO_ID", "NAME", "B01003_001E"],
        dtype=str,
        low_memory=False
    )

    d = d.rename(columns={"B01003_001E": "population"})
    d["year"] = year

    dfs.append(d)

acs_pop = pd.concat(dfs, ignore_index=True)

# Convert to numeric
acs_pop["population"] = pd.to_numeric(acs_pop["population"], errors="coerce")

# Extract ZIP (ZCTA)
acs_pop["zip"] = acs_pop["NAME"].str.extract(r"(\d{5})")
acs_pop["zip"] = acs_pop["zip"].astype(str).str.zfill(5)

acs_pop = acs_pop.sort_values(["zip", "year"]).reset_index(drop=True)

acs_pop.head()

,GEO_ID,NAME,population,year,zip
0,Geography,Geographic Area Name,NaN,2013,00nan
1,Geography,Geographic Area Name,NaN,2014,00nan
2,Geography,Geographic Area Name,NaN,2015,00nan
3,Geography,Geographic Area Name,NaN,2016,00nan
4,Geography,Geographic Area Name,NaN,2017,00nan


## Load Income Data

In [15]:
import pandas as pd
import glob
import re
from pathlib import Path

folder = r"C:\Users\minht\Capstone new\incomenew"

files = glob.glob(str(Path(folder) / "ACSDT5Y*.B19013-Data.csv"))

dfs = []

for f in files:
    
    year = int(re.search(r"ACSDT5Y(\d{4})\.B19013-Data\.csv", Path(f).name).group(1))

    d = pd.read_csv(
        f,
        usecols=["GEO_ID", "NAME", "B19013_001E"],
        dtype=str,
        low_memory=False
    )

    d = d.rename(columns={"B19013_001E": "median_income"})
    d["year"] = year

    dfs.append(d)

acs_income = pd.concat(dfs, ignore_index=True)

# Convert to numeric
acs_income["median_income"] = pd.to_numeric(acs_income["median_income"], errors="coerce")

# Extract ZIP from NAME (ZCTA format)
acs_income["zip"] = acs_income["NAME"].str.extract(r"(\d{5})")
acs_income["zip"] = acs_income["zip"].astype(str).str.zfill(5)

acs_income = acs_income.sort_values(["zip", "year"]).reset_index(drop=True)

acs_income.head()

,GEO_ID,NAME,median_income,year,zip
0,Geography,Geographic Area Name,NaN,2013,00nan
1,Geography,Geographic Area Name,NaN,2014,00nan
2,Geography,Geographic Area Name,NaN,2015,00nan
3,Geography,Geographic Area Name,NaN,2016,00nan
4,Geography,Geographic Area Name,NaN,2017,00nan


## Merge all data by zip-code:

In [16]:
# Merge with WFH data
zip_year_ca = zip_year_ca.merge(
    acs_wfh[["zip", "year", "wfh_share"]],
    left_on=["zip_code", "year"],
    right_on=["zip", "year"],
    how="left"
)
zip_year_ca = zip_year_ca.drop(columns=["zip"])

In [18]:
# Merge with Population data
zip_year_ca = zip_year_ca.merge(
    acs_pop[["zip", "year", "population"]],
    left_on=["zip_code", "year"],
    right_on=["zip", "year"],
    how="left"
).drop(columns=["zip"])

In [19]:
# Merge with income data
zip_year_ca = zip_year_ca.merge(
    acs_income[["zip", "year", "median_income"]],
    left_on=["zip_code", "year"],
    right_on=["zip", "year"],
    how="left"
).drop(columns=["zip"])

In [20]:
# Check the quality of merging data:
zip_year_ca[["wfh_share","population","median_income"]].isna().mean()

wfh_share        0.004036
population       0.003253
median_income    0.021688
dtype: float64

In [21]:
# Visualize data: 
zip_year_ca.head()

,RegionID,zip_code,State,City,CountyName,year,price,wfh_share,population,median_income
0,95982,90001,CA,Florence-Graham,Los Angeles County,2013,184912.712009,64.6,54760.0,35097.0
1,95982,90001,CA,Florence-Graham,Los Angeles County,2014,225886.255973,64.1,56314.0,34050.0
2,95982,90001,CA,Florence-Graham,Los Angeles County,2015,253188.792513,63.8,57227.0,33887.0
3,95982,90001,CA,Florence-Graham,Los Angeles County,2016,273868.654666,65.5,57942.0,34323.0
4,95982,90001,CA,Florence-Graham,Los Angeles County,2017,315517.525062,66.0,58738.0,35660.0


In [23]:
zip_year_ca[[
    "price",
    "wfh_share",
    "median_income",
    "population"
]].isna().mean()

price            0.023857
wfh_share        0.004036
median_income    0.021688
population       0.003253
dtype: float64

In [25]:
import numpy as np

zip_year_ca["log_price"]  = np.log(zip_year_ca["price"])
zip_year_ca["log_income"] = np.log(zip_year_ca["median_income"])
zip_year_ca["log_pop"]    = np.log1p(zip_year_ca["population"])  # safe if zero exists

In [27]:
reg_vars = ["log_price", "wfh_share", "log_income", "log_pop", "zip_code", "year"]

reg_df = zip_year_ca.dropna(subset=reg_vars).copy()

reg_df.shape

(15868, 13)

In [33]:
import numpy as np

summary = reg_df[[
    "log_price",
    "wfh_share",
    "median_income",
    "population"
]].agg([
    "count",
    "mean",
    "std",
    "min",
    "median",
    "max"
]).T

summary

,count,mean,std,min,median,max
log_price,15868.0,13.082832,0.681897,10.560854,13.103557,15.50641
wfh_share,15868.0,72.081447,11.957440,0.000000,74.900000,100.00000
median_income,15868.0,77140.890471,35417.548010,6827.000000,69961.000000,247604.00000
population,15868.0,26549.123393,22100.672229,10.000000,24490.500000,111165.00000


In [37]:
import statsmodels.formula.api as smf

model = smf.ols(
    "log_price ~ wfh_share + median_income + population + C(zip_code) + C(year)",
    data=reg_df
).fit(
    cov_type="cluster",
    cov_kwds={"groups": reg_df["zip_code"]}
)

print(model.summary())

C:\Users\minht\anaconda3\Lib\site-packages\statsmodels\base\model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 1513, but rank is 13
  warnings.warn('covariance of constraints does not have full '


                            OLS Regression Results                            
Dep. Variable:              log_price   R-squared:                       0.990
Model:                            OLS   Adj. R-squared:                  0.989
Method:                 Least Squares   F-statistic:                 1.442e+05
Date:                Fri, 20 Feb 2026   Prob (F-statistic):               0.00
Time:                        21:28:16   Log-Likelihood:                 19873.
No. Observations:               15868   AIC:                        -3.672e+04
Df Residuals:                   14354   BIC:                        -2.510e+04
Df Model:                        1513                                         
Covariance Type:              cluster                                         
                           coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------
Intercept               12.1690 